In [2]:
# step1_preprocess_monpa.py
import pandas as pd
import numpy as np
import monpa
import time
import multiprocessing
from joblib import Parallel, delayed
import warnings
import gc

warnings.filterwarnings('ignore')

def monpa_worker(texts):
    results = []
    for text in texts:
        if pd.isna(text) or str(text).strip() == "":
            results.append("")
            continue
        try:
            # Monpa 斷詞
            words = [item for item in monpa.cut(str(text))]
            # 過濾掉無意義空字串
            words = [w for w in words if len(w.strip()) > 0]
            results.append(" ".join(words))
        except Exception:
            results.append("")
    return results

def process_dataset(filename, output_name, num_cores):
    print(f"📂 讀取 {filename} ...")
    try:
        df = pd.read_csv(filename)
    except:
        print(f"❌ 找不到 {filename}")
        return

    # 合併標題與內文
    df['raw_text'] = df['Column2'].fillna('') + " " + df['Column3'].fillna('')
    raw_texts = df['raw_text'].values
    
    print(f"⚡️ 啟動 Monpa 多核斷詞 ({len(raw_texts)} 筆)...")
    chunks = np.array_split(raw_texts, num_cores)
    
    with Parallel(n_jobs=num_cores, backend='loky', verbose=5) as parallel:
        processed_chunks = parallel(delayed(monpa_worker)(chunk) for chunk in chunks)
    
    df['text_segmented'] = np.concatenate(processed_chunks)
    
    # 只保留需要的欄位存檔
    df[['Column1', 'text_segmented']].to_csv(output_name, index=False, encoding='utf-8-sig')
    print(f"✅ 已儲存至 {output_name}\n")
    
    del df, raw_texts, chunks, processed_chunks
    gc.collect()

if __name__ == '__main__':
    start_time = time.time()
    num_cores = multiprocessing.cpu_count()
    print(f"🚀 [步驟一] Monpa 斷詞預處理 (M4 Pro {num_cores} 核全開)")

    # 處理訓練集
    process_dataset('TrainingData.csv', 'Processed_Train.csv', num_cores)
    
    # 處理測試集
    process_dataset('TestingData.csv', 'Processed_Test.csv', num_cores)
    
    print(f"⏱️ 總耗時: {(time.time() - start_time)/60:.1f} 分鐘")

🚀 [步驟一] Monpa 斷詞預處理 (M4 Pro 12 核全開)
📂 讀取 TrainingData.csv ...
⚡️ 啟動 Monpa 多核斷詞 (11671 筆)...


[Parallel(n_jobs=12)]: Using backend LokyBackend with 12 concurrent workers.
[Parallel(n_jobs=12)]: Done   4 out of  12 | elapsed:   44.2s remaining:  1.5min
[Parallel(n_jobs=12)]: Done   7 out of  12 | elapsed:   45.1s remaining:   32.2s
[Parallel(n_jobs=12)]: Done  10 out of  12 | elapsed:   45.4s remaining:    9.1s
[Parallel(n_jobs=12)]: Done  12 out of  12 | elapsed:   45.7s finished


✅ 已儲存至 Processed_Train.csv

📂 讀取 TestingData.csv ...
⚡️ 啟動 Monpa 多核斷詞 (35604 筆)...


[Parallel(n_jobs=12)]: Using backend LokyBackend with 12 concurrent workers.
[Parallel(n_jobs=12)]: Done   4 out of  12 | elapsed:  2.1min remaining:  4.2min
[Parallel(n_jobs=12)]: Done   7 out of  12 | elapsed:  2.1min remaining:  1.5min
[Parallel(n_jobs=12)]: Done  10 out of  12 | elapsed:  2.2min remaining:   25.9s
[Parallel(n_jobs=12)]: Done  12 out of  12 | elapsed:  2.2min finished


✅ 已儲存至 Processed_Test.csv

⏱️ 總耗時: 3.0 分鐘
